In [ ]:
import os
import torch
import torch.nn as nn
from AIce.functions import testloader,redim
from  AIce.models import NNforNorms

device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

weightlist=[]
for i in os.listdir('./weights'):
    """"Get the list of all the weights"""
    if i[-3:]=='.pt':
        weightlist.append(i)


data,_,_,means,stds,maxes,_=testloader('../data/DRIFT_DATA_TEST.csv',['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm']
                                    ,'buoynorm',trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')# Get the means,stds,maxes using the longest list possible


inputstoweight={}

print(data)
for k,weight in enumerate(weightlist):
    lst=[]

    with open(f'./weights/{weight[:-3]}.txt', 'r') as f:
        """Gets the inputlist associated with a particular Weight"""
        for i in range(2):
            f.readline()
        splits=f.read().split(':')
        lst=splits[1].strip()
        lst=lst.strip('[]').strip()
        lst=lst.split(',')
        lst=[x.strip()[1:-1] for x in lst]

    mlp256= NNforNorms(len(lst)).to(device) # Creates a model for the specific n_inputs
    mlp256.load_state_dict(torch.load(f'./weights/{weight}', weights_only=True))# loads the weights onto the model
    mlp256.eval()  # if you're doing inference, not continuing training
    name=weight[11:19].strip('_') # I wanted a simpler name to differentiate them

    inputstoweight[name]=lst # saves the list of input into a dict
    specificdf=data[lst] # gets the data for a specific inputlist

    out=mlp256(torch.tensor(specificdf.values, dtype= torch.float32).to(device)).to('cpu')# runs the data into the model 
    data[name]=out.detach().numpy() # save the model prediction in the datapd with name associated to n_inputs



Bath size is the full test set
Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
       buoynorm    x_EASE    y_EASE    u_ERA5    v_ERA5   sic_CDR  h_piomas  \
0      1.811923  0.555987  0.485198  0.587591  0.190512  0.996916  2.090739   
1      2.319100  0.491750  0.390600 -0.335793  1.739969  1.000000  3.126633   
2      2.372883  0.512185  0.484238  0.474413  0.435053  0.995579  1.072999   
3      1.909119  0.489006  0.444047 -0.973214 -0.403351  1.000000  2.497801   
4      2.327622  0.447521  0.499113 -0.568250 -0.772593  0.608846  0.995868   
...         ...       ...       ...       ...       ...       ...       ...   
84784  2.596860  0.714442  0.548909 -0.553836  0.969619  0.470374  0.325952   
84785  0.008552  0.359381  0.571497 -1.029456  0.888712  0.974692  0

NameError: name 'device' is not defined